# Stage 4 — Depth-Weighting Ablation

**Goal:** Validate that the depth-weighted consistency loss is meaningfully better
than uniform weighting. Without this ablation, the depth-weighting claim is a
design choice without empirical backing.

**Hypothesis:** Applying higher consistency pressure to later layers (where semantic
content is more abstract and should be stable across instances) produces better
circuit structure than applying equal pressure to all layers.

**Three-way comparison:**
1. `ctls.yaml` — depth-weighted (linear ramp: w_l ∝ l)
2. `ablations/uniform_weighting.yaml` — uniform (w_l = 1/L for all l)
3. `configs/baseline.yaml` — no consistency loss (λ=0)

**What we're measuring:** circuit space silhouette, val accuracy, and per-layer
silhouette profile. If depth-weighting is better, we expect it to be *especially*
better at later layers, with similar or better performance at earlier layers.

## 0. Setup

In [ ]:
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/jacobposchl/model-interpretability'  # <-- edit this once
    REPO_DIR = '/content/ctls'
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
        os.system(f'pip install -r {REPO_DIR}/requirements.txt -q')
    ROOT = REPO_DIR
else:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f'Working directory: {os.getcwd()} ({"Colab" if IN_COLAB else "local"})')


## 1. Visualise the Weight Schemes

Before training anything, confirm what the three weight profiles look like.

In [ ]:
from losses.consistency import _depth_weights

L = 8  # ResNet18 has 8 BasicBlocks
device = torch.device('cpu')

w_linear  = _depth_weights(L, 'linear', device).numpy()
w_uniform = _depth_weights(L, 'uniform', device).numpy()
w_exp     = _depth_weights(L, 'exponential', device).numpy()

fig, ax = plt.subplots(figsize=(8, 4))
x = range(1, L+1)
ax.plot(x, w_linear,  's-', lw=2, color='darkorange',  label='linear (default)')
ax.plot(x, w_uniform, 'o--', lw=2, color='steelblue', label='uniform (ablation)')
ax.plot(x, w_exp,     '^-', lw=2, color='forestgreen', label='exponential')
ax.set_xlabel('Layer index')
ax.set_ylabel('Loss weight w_l (normalised)')
ax.set_title('Depth-weighting schemes for ℒ_cons')
ax.set_xticks(list(x))
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('experiments/ctls/stage4_weight_schemes.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Train Uniform-Weighting Variant

**Skip if you have `experiments/ablations/uniform_weighting/best.pt`.**

All hyperparameters are identical to the full CTLS run — only `weight_scheme`
changes from `linear` → `uniform`.

In [ ]:
from training.trainer import Trainer

with open('configs/ablations/uniform_weighting.yaml') as f:
    config_uniform = yaml.safe_load(f)

trainer_uniform = Trainer(config_uniform)
trainer_uniform.train()

## 3. Load All Three Models

In [ ]:
from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder
from data.cifar import get_standard_loaders

def load_model(config_path, checkpoint_path, device):
    with open(config_path) as f:
        cfg = yaml.safe_load(f)
    mcfg = cfg['model']
    ecfg = mcfg['meta_encoder']
    tcfg = cfg['training']
    soft_mask = SoftMask(init_temperature=tcfg['temperature']['init']).to(device)
    backbone = CTLSBackbone(
        arch=mcfg['arch'], num_classes=mcfg['num_classes'],
        soft_mask=soft_mask, pretrained=False,
    ).to(device)
    meta_encoder = MetaEncoder(
        layer_dims=backbone.layer_dims,
        hidden_dim=ecfg.get('hidden_dim', 256),
        embedding_dim=ecfg.get('embedding_dim', 64),
        encoder_type=ecfg.get('encoder_type', 'mlp'),
    ).to(device)
    ckpt = torch.load(checkpoint_path, map_location=device)
    backbone.load_state_dict(ckpt['backbone_state'])
    meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
    backbone.eval()
    meta_encoder.eval()
    return backbone, meta_encoder, ckpt

bb_depth,   me_depth,   ckpt_depth   = load_model('configs/ctls.yaml',
                                                    'experiments/ctls/best.pt', DEVICE)
bb_uniform, me_uniform, ckpt_uniform = load_model('configs/ablations/uniform_weighting.yaml',
                                                    'experiments/ablations/uniform_weighting/best.pt', DEVICE)
bb_base,    me_base,    ckpt_base    = load_model('configs/baseline.yaml',
                                                    'experiments/baseline/best.pt', DEVICE)

print(f'Depth-weighted: val_acc={ckpt_depth["val_acc"]:.4f}')
print(f'Uniform:        val_acc={ckpt_uniform["val_acc"]:.4f}')
print(f'Baseline:       val_acc={ckpt_base["val_acc"]:.4f}')

with open('configs/ctls.yaml') as f:
    config = yaml.safe_load(f)
dcfg = config['data']
_, val_loader = get_standard_loaders(
    data_dir=dcfg['data_dir'], batch_size=dcfg['batch_size'],
    num_workers=dcfg.get('num_workers', 4), augment=False, download=True,
)

## 4. Circuit Space Silhouette: Three-Way Comparison

In [ ]:
from evaluation.circuit_viz import CircuitVisualizer

viz_depth   = CircuitVisualizer(bb_depth,   me_depth,   val_loader, DEVICE)
viz_uniform = CircuitVisualizer(bb_uniform, me_uniform, val_loader, DEVICE)
viz_base    = CircuitVisualizer(bb_base,    me_base,    val_loader, DEVICE)

print('Computing silhouette scores...')
sil_depth   = viz_depth.cluster_separation_score(max_samples=3000)
sil_uniform = viz_uniform.cluster_separation_score(max_samples=3000)
sil_base    = viz_base.cluster_separation_score(max_samples=3000)

print()
print('=== Circuit Space Silhouette Comparison ===')
print(f"{'Variant':<22} {'Circuit':>10} {'Output':>10} {'Val acc':>10}")
print('-' * 55)
for name, sil, ckpt in [
    ('Baseline (λ=0)', sil_base, ckpt_base),
    ('Uniform weights', sil_uniform, ckpt_uniform),
    ('Depth-weighted', sil_depth, ckpt_depth),
]:
    print(f"{name:<22} {sil['silhouette_circuit']:>10.4f} {sil['silhouette_output']:>10.4f} {ckpt['val_acc']:>10.4f}")

## 5. Per-Layer Silhouette: Where Depth-Weighting Helps

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

def layer_sils(backbone, loader, n=2000):
    all_layers = [[] for _ in range(len(backbone.layer_dims))]
    all_labels = []
    total = 0
    with torch.no_grad():
        for x, labels in loader:
            x = x.to(DEVICE)
            _, traj = backbone(x)
            for i, h in enumerate(traj):
                all_layers[i].append(h.cpu())
            all_labels.append(labels)
            total += x.shape[0]
            if total >= n: break
    layers = [torch.cat(l, 0)[:n].numpy() for l in all_layers]
    labels = torch.cat(all_labels, 0)[:n].numpy()
    scores = []
    for acts in layers:
        a = PCA(n_components=min(50, acts.shape[1])).fit_transform(acts)
        scores.append(silhouette_score(a, labels, metric='euclidean', sample_size=min(1000, n)))
    return scores

print('Layer silhouettes...')
sils_depth   = layer_sils(bb_depth,   val_loader)
sils_uniform = layer_sils(bb_uniform, val_loader)
sils_base    = layer_sils(bb_base,    val_loader)
print('Done.')

In [ ]:
L = len(sils_depth)
x = range(1, L+1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, sils_base,    'o--', color='gray',        lw=2, label='Baseline (λ=0)')
ax.plot(x, sils_uniform, 's--', color='steelblue',   lw=2, label='Uniform weights')
ax.plot(x, sils_depth,   '^-',  color='darkorange',  lw=2.5, label='Depth-weighted (ours)')

ax.set_xlabel('Layer index')
ax.set_ylabel('Silhouette score')
ax.set_title('Stage 4 Ablation: per-layer semantic structure by weighting scheme')
ax.set_xticks(list(x))
ax.axhline(0, color='gray', lw=0.6, linestyle='--')
ax.legend()
ax.grid(True, alpha=0.3)

# Annotate the improvement region
for i, (d, u) in enumerate(zip(sils_depth, sils_uniform)):
    if d > u:
        ax.annotate(f'+{d-u:.3f}', xy=(i+1, max(d, u)),
                    xytext=(i+1, max(d, u)+0.015),
                    ha='center', fontsize=7, color='darkorange')

fig.tight_layout()
os.makedirs('experiments/ctls', exist_ok=True)
fig.savefig('experiments/ctls/stage4_per_layer_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. UMAP: Side-by-Side Visual Comparison

In [ ]:
import umap
import torch.nn.functional as F

def collect_embeddings(backbone, meta_enc, loader, n=3000):
    all_z, all_labels = [], []
    total = 0
    with torch.no_grad():
        for batch in loader:
            x, labels = batch
            x = x.to(DEVICE)
            _, traj = backbone(x)
            z = meta_enc(traj)
            all_z.append(z.cpu())
            all_labels.append(labels)
            total += x.shape[0]
            if total >= n: break
    return torch.cat(all_z, 0)[:n].numpy(), torch.cat(all_labels, 0)[:n].numpy()

print('Collecting embeddings...')
z_depth,   labels = collect_embeddings(bb_depth,   me_depth,   val_loader)
z_uniform, _      = collect_embeddings(bb_uniform, me_uniform, val_loader)
z_base,    _      = collect_embeddings(bb_base,    me_base,    val_loader)

print('Fitting UMAP...')
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30)
xy_depth   = reducer.fit_transform(z_depth)
xy_uniform = umap.UMAP(n_components=2, random_state=42, n_neighbors=30).fit_transform(z_uniform)
xy_base    = umap.UMAP(n_components=2, random_state=42, n_neighbors=30).fit_transform(z_base)

CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
colors = plt.cm.get_cmap('tab10', 10)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Stage 4: Circuit space UMAP by weighting scheme', fontsize=14)

for ax, xy, title in [
    (axes[0], xy_base,    'Baseline (λ=0)'),
    (axes[1], xy_uniform, 'Uniform weights'),
    (axes[2], xy_depth,   'Depth-weighted'),
]:
    for cls in range(10):
        m = labels == cls
        ax.scatter(xy[m,0], xy[m,1], c=[colors(cls)], s=4, alpha=0.5, rasterized=True)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

handles = [plt.Line2D([0],[0],marker='o',color='w',markerfacecolor=colors(i),
           label=CIFAR10_CLASSES[i],markersize=8) for i in range(10)]
fig.legend(handles=handles, loc='lower center', ncol=10, bbox_to_anchor=(0.5, -0.02), fontsize=9)
fig.tight_layout()
fig.savefig('experiments/ctls/stage4_umap_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

| Variant | Circuit sil. | Output sil. | Val acc |
|---------|-------------|------------|--------|
| Baseline (λ=0) | ___ | ___ | ___ |
| Uniform weights | ___ | ___ | ___ |
| Depth-weighted | ___ | ___ | ___ |

**Interpretation:**
- If depth-weighted > uniform on circuit silhouette → depth-weighting claim is supported
- If the gap is small → the depth-weighting is a minor refinement, not a core contribution
- If uniform > depth-weighted → re-examine the w_l schedule (may need to reduce early-layer
  weights more aggressively)

**Next:** Run `stage5_interpretability.ipynb` for SAE-based monosemanticity analysis.